# **STEP 1 : Data Exploration & Cleaning**

<a>- Load and inspect the dataset.</a><BR>
<a>- Handle missing values and remove duplicates entries.</a><BR>
<a>- Convert columns to appropriate data types.</a><BR>
<a>- Perform feature engineering to create new useful variables.</a>


## How we handle missing values

- Date: Delete the line
- Departure station: fill 0
- Arrival station: fill 0
- Average journey time: Mean
- Number of scheduled trains: Interpolate (Trend?)
- Number of cancelled trains: Interpolate (Trend?)
- Cancellation comments: Delete column or fill 0
- Number of trains delayed at departure: Interpolate (Trend?)
- Average delay of late trains at departure: Mean
- Average delay of all trains at departure: Mean
- Departure delay comments: Delete column or fill 0
- Number of trains delayed at arrival: Interpolate (Trend?)
- Average delay of late trains at arrival: Mean
- Average delay of all trains at arrival: Mean
- Arrival delay comments: fill data to 0 since or delete column since we won't need comment to do statistic work
- Number of trains delayed > 15min: Interpolate (Trend?)
- Average delay of trains > 15min (if competing with flights): (Since there is a if, maybe sometime the data is missing for a reason therefore fill it with 0)
- Number of trains delayed > 30min: Mean or Median (Need to know if data distrubition is gaussian distribution or skewed distribution) or interpolate
- Number of trains delayed > 60min: Mean or Median (Need to know if data distrubition is gaussian distribution or skewed distribution) or interpolate
- Pct delay due to external causes: Interpolate (Since the data is a %, there is a trend)
- Pct delay due to infrastructure: Interpolate (Since the data is a %, there is a trend)
- Pct delay due to traffic management: Interpolate (Since the data is a %, there is a trend)
- Pct delay due to rolling stock: Interpolate (Since the data is a %, there is a trend)
- Pct delay due to station management and equipment reuse: Interpolate (Since the data is a %, there is a trend)
- Pct delay due to passenger handling (crowding, disabled persons, connections): Interpolate (Since the data is a %, there is a trend)



#### Import all necessary library

In [1]:
import pandas as pd
from fuzzywuzzy import fuzz
import seaborn as sns
import matplotlib.pyplot as plt

#### Read CSV and file into Dataframe

In [2]:
df = pd.read_csv("dataset.csv", sep=';')

#### Remove duplicates entry, convert types, get absolute numeric numbers and remove everything not useful such as comments.

In [3]:
df.drop_duplicates(inplace=True)
df = df.convert_dtypes()
df.drop("Cancellation comments", axis=1, inplace=True)
df.drop("Departure delay comments", axis=1, inplace=True)
df.drop("Arrival delay comments", axis=1, inplace=True)
df[df.select_dtypes(include='number').columns] = df.select_dtypes(include='number').abs()

#### Replace date in the following format : YEAR-MM

In [4]:
def replace_date(date:str) -> str:
    try:
        len_date = len(date) - 3
        res = date[:len_date] + '-' + date[len_date + 1:]
        return res
    except Exception:
        pass

df['Date'] = df.iloc[:, 0].apply(replace_date)
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m', errors='coerce')

#### Replace empty rown by mean of the current column (only integer value) & Removing years > 2025.

In [5]:
for key in range(4, len(df.keys())):
    try:
        df[df.keys()[key]] = df[df.keys()[key]].fillna(df[df.keys()[key]].mean())
    except Exception:
        pass

df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year
df.dropna(subset='Year', inplace=True)
df.dropna(subset='Month', inplace=True)

## REMOVE undefined years and years which are superior than 2025.
df = df[df['Year'] <= 2025]

#### Replace empty rown by the most used value in **ARRIVAL STATION** and **DEPARTURE STATION**

In [6]:
for key in range(2, 4):
    try:
        df[df.keys()[key]] = df[df.keys()[key]].fillna(df[df.keys()[key]].value_counts().idxmax())
    except Exception:
        pass

#### Replace empty or mistake value in **SERVICE** by **INTERNATIONAL** or **NATIONAL**

In [7]:
## handle Service National or International
def replace_service(service:str) -> str:
    try:
        len_service = len(service)
        note_i = 0
        note_n = 0
        international = "International"
        national = "National"
        for i in range(len_service):
            if service[i] == national[i]:
                note_n += 1
            if service[i] == international[i]:
                note_i += 1
        if (note_i > note_n):
            return international
        return national
    except Exception:
        return "International"

df["Service"] = df.iloc[:, 1].apply(replace_service)

#### Replace mistake in **ARRIVAL STATION** and **DEPARTURE STATION** using the ressemblance

In [8]:
choices = []

def is_int(str:str) -> bool:
    len_str = len(str)
    for i in range(len_str):
        if (str[i].isdigit()):
            return True
    return False

def build_array(array:list) -> list:
    new_array = []
    for i in range(len(array)):
        if (array[i].isupper() and not is_int(array[i])):
            new_array.append(array[i])
    return new_array

def replace_str_station(str:str):
    if (str.isupper() and not is_int(str)):
        return str
    for element in choices:
        if (fuzz.ratio(element, str) >= 80):
            return element

for key in range(2, 4):
    element = df.keys()[key]
    choices = build_array(df[element].values)
    df[element] = df[element].apply(replace_str_station)

count_departure = df['Departure station'].value_counts()
delete = count_departure[count_departure < 4].index
df = df[~df['Departure station'].isin(delete)]

count_arrival = df['Arrival station'].value_counts()
delete = count_arrival[count_arrival < 4].index
df = df[~df['Arrival station'].isin(delete)]

#### Write a CSV through the Dataframe

In [9]:
df.to_csv("cleaned_dataset.csv", sep=";", index=False)

# **STEP 2 : Data Visualization & Analysis**

<a>- Generate summary statistics to understand the dataset better.</a><BR>
<a>- Plot delay distributions and identify common delay durations.</a><BR>
<a>- Compare delays across different stations and times of day.</a><BR>
<a>- Use heatmaps to explore correlations between different variables.</a>

In [10]:
df['Slight delay'] = False
df['Major delay'] = False
for i in range(len(df['Number of trains delayed > 15min'])):
    try:
        df.loc[i, 'Slight delay'] = df['Number of trains delayed > 15min'][i] > 0
        df.loc[i, 'Major delay'] = df['Number of trains delayed > 30min'][i] + df['Number of trains delayed > 60min'][i] > 0
    except Exception:
        pass

# Creation of Routes, (Departure + arrival)
df["Route"] = df["Departure station"] + " --> " + df["Arrival station"]


In [11]:
sns.lineplot(data=df, x="Month", y="Average delay of all trains at departure")
plt.show()
## June, july and August, delays are much more frequent which indicate multiple events, such as Vacation or tourism.

In [12]:
sns.lineplot(data=df, x='Year', y="Average delay of late trains at arrival")
plt.show()
sns.lineplot(data=df, x='Year', y="Average delay of late trains at departure")
plt.show()
## Same as the last graphic, but on a yearly basis.

In [ ]:
top_routes = (df.groupby("Route")["Number of scheduled trains"].sum().sort_values(ascending=False).head(10).reset_index())
sns.barplot(data=top_routes, x='Number of scheduled trains', y='Route')
plt.show()
## Most used routes by trains, which result in more fixes for the routes.

In [ ]:
most_delayed_routes = (df.groupby("Route")["Average delay of all trains at arrival"].mean().sort_values(ascending=False).head(10).reset_index())
sns.barplot(data=most_delayed_routes, x="Average delay of all trains at arrival", y="Route")
plt.show()
## Theses route are a point of stress for the railways since they are the top 10 of most delayed routes.